# Extending modules without inheritance: hooks in Starsim

**Starsim show and tell** — Robyn Stuart, 2026-03-31

How we used callback hooks to let FetalHealth plug into Pregnancy — and when you might (or might not) want this pattern.

---
## 1. The Starsim loop — order of operations

Every timestep, Starsim runs a **fixed sequence** of 16 operations. The ordering is intentional — e.g., networks update before disease transmission so new partners can transmit this timestep.

![Loop order of operations](loop_order.png)

In [ ]:
import starsim as ss
import matplotlib.pyplot as plt

# Build a simple sim and inspect the loop plan
sim = ss.Sim(diseases='sir', networks='random', demographics=True)
sim.init()
sim.loop.plot(simplify=True)

The key ordering (for the steps that do the interesting work) is:

| Step | What happens |
|------|-------------|
| 3 | **Custom modules** run |
| 4 | **Demographics** — births, deaths, pregnancy |
| 5 | **Disease state changes** — e.g. exposed → infected |
| 6 | **Connectors** between diseases |
| 7 | **Networks** — add/remove edges |
| 8 | **Interventions** — vaccination, treatment, etc. |
| 9 | **Disease transmission** — over current networks |

---
## 2. What if you want to extend a module?

Say you have a basic SIR disease model, and now you want to track **viral load** during infection. Or you have a Pregnancy module and you want to track **fetal health** outcomes.

The new logic needs to run at specific moments *inside* the existing module's step — not before or after it in the loop. For example, FetalHealth needs to:

1. **Initialize fetal state** right when a woman conceives (partway through `Pregnancy.step()`)
2. **Classify birth outcomes** right when she delivers (also partway through `Pregnancy.step()`)

There are two natural approaches: **inheritance** and **hooks**.

---
## 3. Two approaches

![Inheritance vs hooks](hooks_vs_inheritance.png)

### Approach A: inheritance

Subclass the module and override its methods:

```python
class SIRWithViralLoad(ss.SIR):
    def step_state(self):
        super().step_state()          # Run the original
        self.update_viral_load()      # Add new logic

    def set_prognoses(self, uids, sources=None):
        super().set_prognoses(uids, sources)
        self.viral_load[uids] = self.pars.initial_vl.rvs(uids)
```

For something like viral load, this is probably the right call — viral load *is* part of the disease. It makes sense for it to live in the same class, accessed as `sim.diseases.sir.viral_load`.

### Approach B: hooks (callbacks)

Alternatively, the base module can expose **hooks** — callback lists that fire at key moments. To support this, you'd write SIR with hook points:

```python
class SIR(Infection):
    def __init__(self, ...):
        ...
        self._infection_callbacks = []

    def add_infection_callback(self, fn):
        self._infection_callbacks.append(fn)

    def set_prognoses(self, uids, sources=None):
        super().set_prognoses(uids, sources)
        ...
        # Fire hooks after new infections are set up
        for cb in self._infection_callbacks:
            cb(uids)
```

Then an external module can register with the hook:

```python
class ViralLoadTracker(ss.Module):
    def init_pre(self, sim):
        super().init_pre(sim)
        sim.diseases.sir.add_infection_callback(self.on_infection)

    def on_infection(self, uids):
        self.viral_load[uids] = self.pars.initial_vl.rvs(uids)
```

You probably wouldn't actually do this for viral load — `sim.diseases.sir.viral_load` is a much more natural home than a separate tracker module.

### So when *do* hooks make sense?

Hooks make sense when the extension **belongs to a different entity** than the base module. This is exactly the case for Pregnancy + FetalHealth:

- `Pregnancy` tracks things about the **mother**: gestation, parity, delivery timing
- `FetalHealth` tracks things about the **infant**: birth weight, LBW, SGA

It wouldn't make sense for `sim.demographics.pregnancy.lbw` to be a property of the Pregnancy module — low birth weight is a property of the baby, not the pregnancy. But the information needed to *compute* it (when conception happens, when delivery happens) only exists inside `Pregnancy.step()`.

Hooks solve this: Pregnancy exposes the right moments, and FetalHealth uses them to track infant outcomes separately.

---
## 4. Hooks in practice: Pregnancy + FetalHealth

This is exactly the pattern we used for fetal health tracking. The Pregnancy module defines two hooks:

```python
# In Pregnancy.__init__()
self._conception_callbacks = []
self._delivery_callbacks = []
```

And fires them at the right moments:

```python
# After making pregnancies:
for cb in self._conception_callbacks:
    cb(conceivers)

# After processing deliveries:
for cb in self._delivery_callbacks:
    cb(mother_uids, newborn_uids)
```

FetalHealth registers at init and then gets called at exactly the right moments:

```python
# In FetalHealth.init_pre()
preg = sim.demographics.pregnancy
preg.add_conception_callback(self.on_conception)
preg.add_delivery_callback(self.on_delivery)
```

![Inside Pregnancy.step()](pregnancy_hooks.png)

### Chaining

Hooks are composable — FetalHealth itself exposes `add_conception_callback()`, so **disease connectors** can hook in too:

```
Pregnancy.step()
  └─> FetalHealth.on_conception()
        └─> SyphilisConnector.on_fetal_exposure()
```

Each layer adds complexity without modifying the layer below.

---
## 5. Minimal example

A tiny `BirthWeightTracker` that hooks into Pregnancy — the same pattern FetalHealth uses, stripped to essentials.

In [ ]:
import starsim as ss
import numpy as np
import matplotlib.pyplot as plt


class BirthWeightTracker(ss.Module):
    """Minimal example: track birth weights via Pregnancy hooks."""

    def __init__(self, **kwargs):
        super().__init__(name='birth_weight_tracker')
        self.define_states(
            ss.FloatArr('weight_percentile'),
            ss.FloatArr('birth_weight'),
        )
        self.weights_at_birth = []  # Store for plotting
        return

    def init_pre(self, sim):
        super().init_pre(sim)
        # Register our hooks with Pregnancy
        preg = sim.demographics.pregnancy
        preg.add_conception_callback(self.on_conception)
        preg.add_delivery_callback(self.on_delivery)
        return

    def on_conception(self, uids):
        """Called by Pregnancy when new pregnancies begin."""
        self.weight_percentile[uids] = np.random.normal(loc=1.0, scale=0.1, size=len(uids))
        print(f'  t={self.sim.now}: {len(uids)} new pregnancies — assigned weight percentiles')
        return

    def on_delivery(self, mother_uids, newborn_uids):
        """Called by Pregnancy when deliveries occur."""
        if not len(newborn_uids):
            return
        parents = self.sim.people.parent[newborn_uids]
        bw = 3400 * self.weight_percentile[parents]  # Simple: baseline * percentile
        self.birth_weight[newborn_uids] = bw
        self.weights_at_birth.extend(bw.tolist())
        print(f'  t={self.sim.now}: {len(newborn_uids)} births — mean weight {bw.mean():.0f}g')
        return

    def step(self):
        pass  # All logic is driven by hooks!


# Run it
sim = ss.Sim(
    dur = 10,
    demographics = [ss.Pregnancy(fertility_rate=50), ss.Deaths(death_rate=10)],
    modules = BirthWeightTracker(),
    networks = ss.PrenatalNet(),
)
sim.run()

In [ ]:
# Plot the birth weight distribution
tracker = sim.get_module('birth_weight_tracker')
if tracker.weights_at_birth:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(tracker.weights_at_birth, bins=25, color='#ec4899', alpha=0.7, edgecolor='white')
    ax.axvline(2500, color='red', linestyle='--', lw=2, label='LBW threshold (2500g)')
    ax.set_xlabel('Birth weight (g)')
    ax.set_ylabel('Count')
    ax.set_title('Birth weights from hook-driven tracking')
    ax.legend()
    plt.show()
else:
    print('No births occurred — try increasing fertility_rate or dur')

Notice that `BirthWeightTracker.step()` does **nothing** — all the work happens inside `Pregnancy.step()` via hooks. The tracker's `on_conception` and `on_delivery` methods fire at exactly the right moments, with exactly the right UIDs.

This is the same pattern the real `ss.FetalHealth` module uses, just stripped down to the essentials.

---
## Which approach is right?

| | **Inheritance** | **Hooks** |
|---|---|---|
| **Extension is...** | Intrinsic to the module (viral load *is* part of the disease) | A separate concern (fetal health is a *lens* on pregnancy) |
| **Base module** | Modified (subclassed) | Unchanged |
| **Composability** | Hard to stack multiple extensions | Multiple hooks compose naturally |
| **Coupling** | Tight — must track parent's internals | Loose — only depends on the hook signature |
| **Overhead** | None — standard Python | Base module must define hooks upfront |
| **Best when** | One team owns both base and extension | Different teams/projects extend independently |
| **Example** | `SIRWithViralLoad(SIR)` | `FetalHealth` → `Pregnancy` |

Neither is strictly better — it depends on whether the extension is a core part of the module or an independent concern that should be pluggable.